In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")
paths = os.path.join(path, 'RegFood.csv')
print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from sklearn.preprocessing import  OneHotEncoder
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score,KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score



In [ ]:

pokemon_path = os.path.join(path, 'Q1_data.csv')
df =pd.read_csv(pokemon_path)

print(f"Shape: {df.shape}")


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:


def check_target_distribution(df, target_column):
    df[target_column].hist(bins=30, edgecolor='black')

    plt.title(f"Target Distribution ({target_column})")
    plt.xlabel(target_column)
    plt.ylabel("Frequency")
    # plt.grid(False)

    plt.show()

check_target_distribution(df,'Delivery_Time')

In [ ]:
# Task 1: Write your code here:
df=df.drop(columns='Order_ID')
df.head(2)

In [ ]:
# Task 2: Write your code here:
# 2. Do we have missing values?

#----code for check:----
# missing_percentage = (df.isnull().sum() / len(df)) * 100
# missing_data = pd.DataFrame({
#     'Column': missing_percentage.index,
#     'Missing_Percentage': missing_percentage.values
# })
# missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

# print("Missing Data Analysis:")
# missing_data

#we can drop it cuz is small
cols=['Delivery_Time','Weather','Traffic_Level','Time_of_Day','Courier_Experience_yrs']
df=df.dropna(subset=cols)


In [ ]:
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:

categorical_cols = df.select_dtypes(include=["object"]).columns
label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le
df.info()

In [ ]:
# Task 5: Write your code here:
scaler=StandardScaler()
features = df.columns.drop("Delivery_Time")

df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 6: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('time Distribution')
plt.xlabel('time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.metrics import  mean_absolute_error
n_splits = 5  # K=5 Folds
mae=[]
model=RandomForestRegressor(n_estimators=200)
# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
  model.fit(X_train, y_train)

    # Predict
  y_pred = model.predict(X_test)
  loss=mean_absolute_error(y_test, y_pred)
    # Calculate metrics
  mae.append(loss)
  print(f' MAE for {fold_idx + 1}:{loss}')

print(f"avg sorec:{np.mean(np.array(mae))}")


In [ ]:
# Task 1: Write your code here:
cof=model.feature_importances_
i=0
for feature in features:
 plt.barh(feature, cof[i])
 plt.title(f"{feature} Coefficients")
 plt.xlabel("Coefficient Value (Impact)")
 i+=1

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=50, edgecolor='black')
plt.title('Predicted time Distribution')
plt.xlabel('time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here:
models = {
  "LightGBM": LGBMRegressor(verbose=-1),
  "Decision Tree Regressor": DecisionTreeRegressor(max_depth=10)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)


    # Store results
    loss=mean_absolute_error(y_test, y_pred)
    # Calculate metrics
    mae.append(loss)
    print(f' MAE for {fold_idx + 1}:{loss}')

print(f"avg sorec:{np.mean(np.array(mae))}")